In [29]:
import pandas as pd 
import numpy as np
import re
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display
from astroquery.simbad import Simbad

In [30]:
EXPLICIT_ALIASES = {
    'AUMic': ['AU Mic'],
    'EpsilonIndi': ['eps Ind', 'Epsilon Indi'],
    'piMensae': ['pi Men', 'Pi Mensae'],
}


def candidate_names(name):
    name = str(name).strip()
    candidates = [name]
    candidates.extend(EXPLICIT_ALIASES.get(name, []))

    patterns = [
        (r'^HD(\\d+)$', r'HD \\1'),
        (r'^HIP(\\d+)$', r'HIP \\1'),
        (r'^GJ(\\d+)$', r'GJ \\1'),
        (r'^(55)CncA$', r'55 Cnc'),
    ]

    for pattern, replacement in patterns:
        if re.match(pattern, name, flags=re.IGNORECASE):
            candidates.append(re.sub(pattern, replacement, name, flags=re.IGNORECASE))

    seen = set()
    ordered_candidates = []
    for candidate in candidates:
        if candidate not in seen:
            seen.add(candidate)
            ordered_candidates.append(candidate)

    return ordered_candidates


def fetch_gaia_dr3_id(name):
    for candidate in candidate_names(name):
        ids = Simbad.query_objectids(candidate)
        if ids is None:
            continue

        for row in ids:
            object_id = str(row[0])
            if object_id.startswith('Gaia DR3 '):
                return pd.Series({
                    'matched_name': candidate,
                    'gaia_dr3_id': object_id.replace('Gaia DR3 ', '')
                })

    return pd.Series({
        'matched_name': pd.NA,
        'gaia_dr3_id': pd.NA
    })


#stars_df[['matched_name', 'gaia_dr3_id']] = stars_df['star'].apply(fetch_gaia_dr3_id)



In [31]:
SWEETCAT = False

if SWEETCAT:

    sample_df = pd.read_csv('sample.csv')
    sample_df.head(5)

    sweetcat_url = 'https://sweetcat.iastro.pt/catalog/SWEETCAT_Dataframe.csv'
    sweetcat_df = pd.read_csv(sweetcat_url)

    # Keep Gaia IDs numeric while still allowing missing values.
    sample_df['gaia_dr3_id'] = pd.to_numeric(sample_df['gaia_dr3_id'], errors='coerce').astype('Int64')
    sweetcat_df['gaia_dr3'] = pd.to_numeric(sweetcat_df['gaia_dr3'], errors='coerce').astype('Int64')

    sweetcat_columns = [
        'Name', 'gaia_dr3', 'Teff', 'eTeff', 'Logg', 'eLogg', '[Fe/H]', 'e[Fe/H]',
        'Vt', 'eVt', 'Gmag', 'Plx', 'Distance', 'Mass_t', 'Radius_t', 'SWFlag',
        'Reference', 'Link', 'Update', 'Database'
    ]

    sample_sweetcat_df = sample_df.merge(
        sweetcat_df[sweetcat_columns],
        left_on='gaia_dr3_id',
        right_on='gaia_dr3',
        how='left'
    )

    selected_columns =['star', 'Name', 'gaia_dr3', 'Teff',
       'eTeff', 'Logg', 'eLogg', '[Fe/H]', 'e[Fe/H]', 'Vt', 'eVt', 'Gmag',
       'Plx', 'Distance', 'Mass_t', 'Radius_t', 'SWFlag', 'Reference', 'Link',
       'Update', 'Database']
    
    sample_sweetcat_df = sample_sweetcat_df[selected_columns]
    sample_sweetcat_df.to_csv('sample_sweetcat.csv', index=False)


In [32]:
sample_df = pd.read_csv('sample_sweetcat.csv')
sample_df.head(2)

,star,Name,gaia_dr3,Teff,eTeff,Logg,eLogg,[Fe/H],e[Fe/H],Vt,...,Gmag,Plx,Distance,Mass_t,Radius_t,SWFlag,Reference,Link,Update,Database
0,55CncA,55 Cnc,704967037090946688,5363.0,59.0,4.29,0.14,0.33,0.04,1.0,...,5.7327,79.4482,12.586818,0.922748,0.901203,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA
1,AUMic,AU Mic,6794047652729201024,3700.0,100.0,NaN,NaN,NaN,NaN,NaN,...,7.8434,102.9432,9.714095,0.512628,0.697741,0,Plavchan et al.2020,https://ui.adsabs.harvard.edu/abs/2020Natur.58...,2020-07-08,EU;NASA


In [33]:
observation_files = sorted(Path('observations/RM').glob('*.rdb'))


def get_obs_names(star_name):
    star_name = str(star_name)
    return [file.name for file in observation_files if file.stem.startswith(f'{star_name}_')]


sample_df['obs_name'] = sample_df['star'].apply(get_obs_names)
sample_df[['star', 'obs_name']].head()

,star,obs_name
0,55CncA,"[55CncA_ESPRESSO18_1.rdb, 55CncA_ESPRESSO19_1...."
1,AUMic,"[AUMic_ESPRESSO19_1.rdb, AUMic_ESPRESSO19_2.rdb]"
2,EpsilonIndi,[]
3,GJ436,[GJ436_ESPRESSO18_1.rdb]
4,HAT-P-26,[HAT-P-26_ESPRESSO19_1.rdb]


In [34]:
observation_files = sorted(Path('observations/Good_RM').glob('*.rdb'))
observation_files[0]

PosixPath('observations/Good_RM/AUMIC_esp19_1.rdb')

In [43]:
file_path = Path('observations/Best_RM')


def read_rdb_file(file_path):
    rdb_df = pd.read_csv(file_path, sep='\t', skiprows=[1])

    for column in ['rjd', 'vrad', 'svrad']:
        rdb_df[column] = pd.to_numeric(rdb_df[column], errors='coerce')

    return rdb_df


def plot_vrad_vs_rjd(obs_name):
    obs_path = file_path / obs_name
    rdb_df = read_rdb_file(obs_path)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=rdb_df['rjd'],
            y=rdb_df['vrad'],
            error_y=dict(type='data', array=rdb_df['svrad'], visible=True),
            mode='markers+lines',
            marker=dict(size=7),
            name=obs_name,
            text=rdb_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>'
        )
    )

    fig.update_layout(
        title=f'vrad vs rjd: {obs_name}',
        xaxis_title='rjd',
        yaxis_title='vrad',
        template='plotly_white',
        hovermode='closest'
    )

    fig.show()

In [44]:
all_obs_names = sorted(path.name for path in file_path.glob('*.rdb'))
obs_selector = widgets.Dropdown(options=all_obs_names, description='obs_name:')
interactive_plot = widgets.interactive_output(plot_vrad_vs_rjd, {'obs_name': obs_selector})
display(obs_selector, interactive_plot)

Dropdown(description='obs_name:', options=('HD189733_esp19_3.rdb', 'HD189733_esp19_4.rdb', 'HD209458_esp19_1.r…

Output()

In [37]:
def save_rdb_rjd_range(input_obs_name, rjd_min, rjd_max, output_obs_name):
    input_path = Path('obs_split_by_night') / input_obs_name
    output_path = Path('obs_split_by_night') / output_obs_name

    with input_path.open() as file_handle:
        header_line = file_handle.readline().rstrip('\n')
        separator_line = file_handle.readline().rstrip('\n')

    rdb_df = read_rdb_file(input_path)
    filtered_df = rdb_df[(rdb_df['rjd'] >= rjd_min) & (rdb_df['rjd'] <= rjd_max)].copy()

    with output_path.open('w') as file_handle:
        file_handle.write(header_line + '\n')
        file_handle.write(separator_line + '\n')
        filtered_df.to_csv(file_handle, sep='\t', index=False, header=False)

    print(f'Saved {len(filtered_df)} rows to {output_path}')
    return filtered_df

In [38]:
#filtered_df = save_rdb_rjd_range(
#    input_obs_name='55CncA_ESPRESSO19_1.rdb',
#    rjd_min=59236.,
#    rjd_max=59237,
#    output_obs_name='../obs_to_be_used/55CncA_esp19_2.rdb'
#)
#print (filtered_df.shape)
#filtered_df.head(2)

In [39]:
split_input_dir = Path('observations/RM')
split_output_dir = Path('obs_split_by_night')


def build_split_prefix(input_obs_name):
    input_path = Path(input_obs_name)
    stem = input_path.stem

    match = re.match(r'(.+)_ESPRESSO(\d+)_\d+$', stem)
    if match is None:
        raise ValueError(f'Could not parse star name and ESPRESSO run from {input_obs_name}')

    star_name = re.sub(r'[^A-Za-z0-9]', '', match.group(1)).upper()
    espresso_run = match.group(2)
    return f'{star_name}_esp{espresso_run}'


def split_rdb_by_night(input_obs_name, gap_days=1.0, input_dir=None, output_dir=None, output_prefix=None):
    input_dir = Path(input_dir) if input_dir is not None else split_input_dir
    output_dir = Path(output_dir) if output_dir is not None else split_output_dir
    input_path = input_dir / input_obs_name
    output_dir.mkdir(parents=True, exist_ok=True)

    with input_path.open() as file_handle:
        header_line = file_handle.readline().rstrip('\n')
        separator_line = file_handle.readline().rstrip('\n')

    rdb_df = read_rdb_file(input_path).sort_values('rjd').reset_index(drop=True)
    group_index = rdb_df['rjd'].diff().gt(gap_days).fillna(False).cumsum()
    output_prefix = output_prefix or build_split_prefix(input_obs_name)

    saved_files = []
    for split_number, (_, split_df) in enumerate(rdb_df.groupby(group_index), start=1):
        output_name = f'{output_prefix}_{split_number}.rdb'
        output_path = output_dir / output_name

        with output_path.open('w') as file_handle:
            file_handle.write(header_line + '\n')
            file_handle.write(separator_line + '\n')
            split_df.to_csv(file_handle, sep='\t', index=False, header=False)

        saved_files.append(output_path)
        print(f'Saved {len(split_df)} rows to {output_path}')

    return saved_files


def split_all_rdb_by_night(gap_days=1.0, input_dir=None, output_dir=None):
    input_dir = Path(input_dir) if input_dir is not None else split_input_dir
    output_dir = Path(output_dir) if output_dir is not None else split_output_dir
    all_saved_files = {}
    for input_path in sorted(input_dir.glob('*.rdb')):
        all_saved_files[input_path.name] = split_rdb_by_night(
            input_obs_name=input_path.name,
            gap_days=gap_days,
            input_dir=input_dir,
            output_dir=output_dir
        )
    return all_saved_files

In [40]:
#split_rdb_by_night(
#    input_obs_name='AUMic_ESPRESSO19_2.rdb',
#    input_dir='observations/RM',
#    output_dir='obs_split_by_night/eps_ili'
#)


In [41]:
# Example for one file:
#saved_files = split_rdb_by_night('AUMic_ESPRESSO19_2.rdb')
# saved_files

# Example for all files:
#all_saved_files = split_all_rdb_by_night()

In [42]:
def delete_short_rdb_files(folder_path, min_total_lines=12):
    folder_path = Path(folder_path)
    deleted_files = []
    kept_files = []

    for file_path in sorted(folder_path.glob('*.rdb')):
        with file_path.open() as file_handle:
            line_count = sum(1 for _ in file_handle)

        if line_count < min_total_lines:
            file_path.unlink()
            deleted_files.append(file_path)
            print(f'Deleted {file_path} because it has {line_count} total lines')
        else:
            kept_files.append(file_path)

    print(f'Kept {len(kept_files)} files and deleted {len(deleted_files)} files')
    return deleted_files

# Example:
#deleted_files = delete_short_rdb_files('obs_split_by_night/', min_total_lines=12)
#deleted_files